# Example 1.5 / 2: Several Objectives

Companion notebook for **1.5 Objectives** in the AMPL lab Documentation, section "Several Objectives". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Several Objectives

A model may declare more than one objective. AMPL sends them all, and the solver optimizes the
**first** one unless told otherwise; the `objective` command selects another. Here the model
also declares `Total_Tons`, the total amount made, and coils are more profitable than in the
example above, so the two objectives have different best plans.

amplpy has no method of its own for the `objective` command, so `run.py` sends it with
`ampl.eval("objective Total_Tons;")`. An objective that is not being optimized still has a
value: the last line prints the profit of the plan that makes the most tons.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                    # products
param rate {PROD} > 0;       # tons produced per hour
param profit {PROD};         # profit per ton
param market {PROD} >= 0;    # most that can be sold, in tons
param avail >= 0;            # hours available

var Make {p in PROD} >= 0, <= market[p];   # tons to make

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];   # first: optimized by default
maximize Total_Tons: sum {p in PROD} Make[p];                 # total tons made, all products

subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;   # hours used <= available

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["avail"] = 40
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 40}
ampl.param["market"] = {"bands": 6000, "coils": 2800}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.var["Make"].to_dict())

ampl.eval("objective Total_Tons;")
ampl.solve(solver="highs")
print(ampl.var["Make"].to_dict())
print(ampl.obj["Total_Profit"].value())

HiGHS 1.14.0: optimal solution; objective 212000
0 simplex iterations
0 barrier iterations
Objective = Total_Profit
{'bands': 4000, 'coils': 2800}
HiGHS 1.14.0: optimal solution; objective 7400
1 simplex iterations
0 barrier iterations
{'bands': 6000, 'coils': 1400}
206000.0


### Pure AMPL: data.dat and model.run

With no `objective` command, the solver reports which objective it used (`Objective =
Total_Profit`). The selection stays in force for every later solve, until another `objective`
command changes it.

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param avail := 40;
param:  rate  profit  market :=
  bands  200    25     6000
  coils  140    40     2800 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Make;
objective Total_Tons;
solve;
display Make, Total_Profit;

HiGHS 1.14.0: optimal solution; objective 212000
0 simplex iterations
0 barrier iterations
Objective = Total_Profit
Make [*] :=
bands  4000
coils  2800
;

HiGHS 1.14.0: optimal solution; objective 7400
1 simplex iterations
0 barrier iterations
Make [*] :=
bands  6000
coils  1400
;

Total_Profit = 206000



### Another way in amplpy

Another way is to set an objective aside: `ampl.obj["Total_Profit"].drop()` makes AMPL ignore
it, so the next solve optimizes `Total_Tons`, and `restore()` brings it back. These are the
`drop Total_Profit;` and `restore Total_Profit;` commands of pure AMPL. (An `objective` command
is the same as dropping every objective and then restoring the one it names.)

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["avail"] = 40
ampl.param["rate"] = {"bands": 200, "coils": 140}
ampl.param["profit"] = {"bands": 25, "coils": 40}
ampl.param["market"] = {"bands": 6000, "coils": 2800}

In [11]:
# run.py
ampl.obj["Total_Profit"].drop()
ampl.solve(solver="highs")
print(ampl.var["Make"].to_dict())

ampl.obj["Total_Profit"].restore()
ampl.solve(solver="highs")
print(ampl.var["Make"].to_dict())

HiGHS 1.14.0: optimal solution; objective 7400
0 simplex iterations
0 barrier iterations
{'bands': 6000, 'coils': 1400}
HiGHS 1.14.0: optimal solution; objective 212000
1 simplex iterations
0 barrier iterations
Objective = Total_Profit
{'bands': 4000, 'coils': 2800}
